# Financial Risk Assessment & Credit Scoring

## 🏦 Business Context

Financial institutions must accurately assess the risk of borrower default to maintain a healthy loan portfolio. This analysis develops a credit scoring model to predict the probability of default (PD) and assigns risk grades, enabling data-driven lending decisions and risk-based pricing.

## 📊 Objectives

1. Analyze borrower demographic and financial data
2. Build a predictive model for loan default (Logistic Regression / XGBoost)
3. Develop a Credit Scorecard (Points-based system)
4. Evaluate model discrimination (ROC-AUC, Gini) and calibration
5. Segment borrowers into risk tiers for strategy formulation

## 🔧 Methodology

- **Data**: Synthetic loan application dataset (Income, Debt, Credit History, etc.)
- **Techniques**: Weight of Evidence (WoE), Information Value (IV), Logistic Regression
- **Metrics**: AUC, Gini Coefficient, KS Statistic

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('RdBu')
%matplotlib inline

print('✓ Libraries loaded successfully')

## 1. Data Generation

Simulating a dataset of 10,000 loan applicants.

In [ ]:
def generate_credit_data(n=10000):
    np.random.seed(42)
    
    # Features
    age = np.random.normal(40, 12, n).astype(int)
    age = np.clip(age, 18, 80)
    
    income = np.random.lognormal(10.5, 0.6, n) # Median ~36k
    debt_to_income = np.random.beta(2, 5, n) * 100 # DTI Ratio
    
    credit_history_years = np.random.exponential(8, n)
    num_open_accounts = np.random.poisson(5, n)
    num_late_payments = np.random.poisson(0.5, n)
    
    employment_length = np.random.choice(['<1 year', '1-3 years', '4-7 years', '>7 years'], n, p=[0.1, 0.3, 0.4, 0.2])
    home_ownership = np.random.choice(['Rent', 'Mortgage', 'Own'], n, p=[0.4, 0.4, 0.2])
    
    # Default Logic (Probabilistic)
    # Higher risk: Low income, High DTI, Late payments, Renting
    logit = -3.0 \
            - 0.00002 * income \
            + 0.05 * debt_to_income \
            + 0.8 * num_late_payments \
            - 0.1 * credit_history_years \
            + (np.where(home_ownership == 'Rent', 0.5, 0))
            
    prob_default = 1 / (1 + np.exp(-logit))
    default = np.random.binomial(1, prob_default)
    
    df = pd.DataFrame({
        'Age': age,
        'Annual_Income': income,
        'DTI_Ratio': debt_to_income,
        'Credit_History_Years': credit_history_years,
        'Num_Open_Accounts': num_open_accounts,
        'Num_Late_Payments': num_late_payments,
        'Employment_Length': employment_length,
        'Home_Ownership': home_ownership,
        'Default': default
    })
    
    return df

df = generate_credit_data()
print(f"Dataset Shape: {df.shape}")
print(f"Default Rate: {df['Default'].mean():.2%}")
display(df.head())

## 2. Exploratory Data Analysis

Comparing features for Defaulters vs Non-Defaulters.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Income Distribution
sns.boxplot(x='Default', y='Annual_Income', data=df, ax=axes[0,0], showfliers=False)
axes[0,0].set_title('Annual Income by Default Status')

# DTI Distribution
sns.kdeplot(data=df, x='DTI_Ratio', hue='Default', fill=True, ax=axes[0,1])
axes[0,1].set_title('Debt-to-Income Ratio Distribution')

# Late Payments
sns.barplot(x='Num_Late_Payments', y='Default', data=df, ax=axes[1,0], ci=None)
axes[1,0].set_title('Default Rate by Late Payments')
axes[1,0].set_ylabel('Probability of Default')

# Home Ownership
sns.barplot(x='Home_Ownership', y='Default', data=df, ax=axes[1,1], ci=None)
axes[1,1].set_title('Default Rate by Home Ownership')

plt.tight_layout()
plt.savefig('outputs/risk_eda.png')
plt.show()

## 3. Preprocessing & Modeling

Training a Logistic Regression model to estimate Probability of Default (PD).

In [ ]:
# Encoding Categorical Variables
df_model = pd.get_dummies(df, drop_first=True)

X = df_model.drop('Default', axis=1)
y = df_model['Default']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Predictions
y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_pred_prob > 0.5).astype(int)

print("Model Trained Successfully")

## 4. Model Evaluation

Assessing discriminatory power using ROC Curve and Gini Coefficient.

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)
gini = 2 * auc - 1

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc:.3f})', color='blue', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
plt.title(f'ROC Curve (Gini = {gini:.3f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True)
plt.savefig('outputs/roc_curve.png')
plt.show()

print(f"AUC Score: {auc:.4f}")
print(f"Gini Coefficient: {gini:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 5. Credit Scorecard Development

Converting probabilities into a standard credit score (e.g., 300-850).

In [ ]:
def calculate_credit_score(prob, min_score=300, max_score=850):
    # Simple linear scaling of log-odds
    # Score = Offset + Factor * ln(odds)
    # Here we map probability directly for simplicity
    return int(max_score - (prob * (max_score - min_score)))

# Apply to test set
scores = [calculate_credit_score(p) for p in y_pred_prob]
df_results = pd.DataFrame({'Probability': y_pred_prob, 'Credit_Score': scores, 'Actual_Default': y_test})

# Visualization of Score Distribution
plt.figure(figsize=(12, 6))
sns.histplot(data=df_results, x='Credit_Score', hue='Actual_Default', bins=30, kde=True, palette={0: 'green', 1: 'red'})
plt.title('Credit Score Distribution: Good (0) vs Bad (1) Loans')
plt.xlabel('Credit Score')
plt.axvline(650, color='black', linestyle='--', label='Cutoff (650)')
plt.legend()
plt.savefig('outputs/score_distribution.png')
plt.show()

## 6. Risk Segmentation & Strategy

Defining risk tiers and approval strategies.

In [ ]:
# Define Risk Tiers
def assign_tier(score):
    if score >= 750: return 'Super Prime'
    elif score >= 700: return 'Prime'
    elif score >= 650: return 'Near Prime'
    elif score >= 600: return 'Subprime'
    else: return 'Deep Subprime'

df_results['Risk_Tier'] = df_results['Credit_Score'].apply(assign_tier)

# Analyze Default Rate by Tier
tier_analysis = df_results.groupby('Risk_Tier').agg({
    'Actual_Default': 'mean',
    'Credit_Score': 'count'
}).rename(columns={'Actual_Default': 'Bad_Rate', 'Credit_Score': 'Count'})

tier_analysis = tier_analysis.reindex(['Super Prime', 'Prime', 'Near Prime', 'Subprime', 'Deep Subprime'])

print("="*60)
print("RISK STRATEGY DASHBOARD")
print("="*60)
print(tier_analysis)

print("\nRECOMMENDED STRATEGY:")
print("1. Super Prime & Prime: Auto-Approve, Lowest Rates.")
print("2. Near Prime: Manual Review or Higher Rate.")
print("3. Subprime: Require Collateral or Co-signer.")
print("4. Deep Subprime: Auto-Decline.")